[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lbutler2405/EMP5027-rows-to-pixels/blob/main/notebooks/practical-3b-time-aware-data-joins/EMP5027-Lecture-3b-Time-Aware-Data-and-Joins.ipynb)


# EMP5027 Lecture 3b: Time-Aware Data and Joins (Parts 3-4)

*EMP5027, Methods in Data Analysis and Quality Assurance*

Dr. Liam Butler | Department of Systems & Control Engineering / Institute of Earth Systems, University of Malta

This notebook continues directly from **Lecture 3: Working with Structured Data** (Parts 1-2: Penguins & Iris). Run this from its own folder as is. It expects a `Parts3-4/` subfolder alongside it containing `Air_Quality.csv` and `Penguins_data.csv`.


## Learning Objectives

By the end of this notebook you will be able to:
- Parse timestamps, index a DataFrame by time, and identify and handle missing time points.
- Impute short gaps, resample to daily means, and compute rolling averages.
- Compare multiple monitoring stations with a multiple linear regression (`statsmodels`).
- Identify and drop highly correlated features.
- Combine tables with joins and concatenation, and export cleaned data plus a metadata dictionary to Excel.



---
# Part 3: Time-Aware Data: Air Quality (NO₂)

Environmental monitoring data almost always comes with a timestamp attached, and that timestamp is not just another column. It is what lets us ask the questions we actually care about: is pollution worse at certain times of day, is a station's sensor dropping readings, and how do different stations compare once we smooth out the noise.

In this part we will parse timestamps properly, check for missing time points, impute short gaps, resample to daily means, compute rolling averages, and compare multiple stations. These are the same building blocks you will use for any sensor network, whether it is measuring air quality, water quality, or soil moisture.



## Running this in Google Colab

Click the badge above to open this notebook directly in Colab, no local setup required. Everything this notebook needs is already available on Colab by default.


In [ ]:
# --- Google Colab setup (safe to run locally too, it just skips this step) ---
import sys

if "google.colab" in sys.modules:
    pass
    print("Running in Colab, ready to go.")
else:
    print("Not running in Colab, assuming packages are already installed locally.")

In [ ]:
import os
os.getcwd()

In [ ]:
os.chdir("Parts3-4")

In [ ]:
os.getcwd()

In [ ]:
import pandas as pd
# Load the dataset
df_aq = pd.read_csv("Air_Quality.csv")
display(df_aq.head())
display(df_aq.info())

In [ ]:
df_aq


## Parse and Index by Time
Right now the `datetime` column is just a string as far as pandas is concerned. We need to convert it to an actual datetime type before pandas can do anything time-aware with it, such as resampling, sorting by time, or computing rolling windows. Let's convert the column and take a look.



In [ ]:
df_aq['datetime'] = pd.to_datetime(df_aq['datetime'])
df_aq.head()

In [ ]:
df_aq.dtypes

In [ ]:
## Our data is in wide format. Let's 'melt' it into a long format.
# Melt the dataframe
df_long = df_aq.melt(
    id_vars='datetime',
    var_name='station',
    value_name='value'
)

print(df_long.head(10))

In [ ]:
len(df_long)

In [ ]:
df_long

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Drop NaN values (boxplots can't plot missing data)
df_long = df_long.dropna(subset=['value'])

# Optional: make cleaner labels
df_long['station'] = df_long['station'].str.replace('station_', '').str.capitalize()
df_long

In [ ]:
# Create boxplot
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_long, x='station', y='value', hue = "station",  palette='Set2')

plt.title('Distribution of Air Quality Measurements per Station')
plt.xlabel('Station')
plt.ylabel('Pollutant Value (µg/m³)')
plt.show()

In [ ]:
# Create boxplot
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_long, x='station', y='value', hue = "station",  palette='Set2')
sns.stripplot(data=df_long, x='station', y='value', color='black', size=5, alpha=0.3)
plt.title('Distribbution of Air Quality Measurements per Station')
plt.xlabel('Station')
plt.ylabel('Pollutant Value (µg/m³)')
plt.show()

In [ ]:
# remove the top and right borders (“spines”), and
# change the font size of your x-axis and y-axis tick labels

plt.figure(figsize=(8, 5))
sns.boxplot(data=df_long, x='station', y='value', hue='station', palette='Set2')
sns.stripplot(data=df_long, x='station', y='value', color='black', size=4, alpha=0.4)

# Titles and labels
plt.title('Distribution of Air Quality Measurements per Station', fontsize=14)
plt.xlabel('Station', fontsize=12)
plt.ylabel('Pollutant Value (µg/m³)', fontsize=12)

# Remove top and right borders
sns.despine(top=True, right=True)

# Change font size of x and y tick labels
plt.xticks(fontsize=15)
plt.yticks(fontsize=15)

plt.tight_layout()

## Save the figure
plt.savefig(
    "air_quality_boxplot.png",   # filename
    dpi=300,                     # resolution: 300 for print-quality
    bbox_inches='tight',         # trims excess white space
    transparent=True             # optional: transparent background
)
plt.show()

In [ ]:
# Shows how environmental time series often have gaps.
import seaborn as sns
sns.heatmap(df_long.pivot(index='datetime', columns='station', values='value').isna(), cbar=False)
plt.title("Missing Data per Station over Time")

In [ ]:
# Shows how environmental time series often have gaps.
sns.heatmap(df_long.pivot(index='datetime', columns='station', values='value').isna(), 
            cmap=sns.color_palette(["white", "black"]),
            cbar=False)
plt.title("Missing Data per Station over Time", fontsize = 12)
plt.xlabel("Station", fontsize = 12)
plt.ylabel("DateTime", fontsize = 12)
plt.tight_layout()
plt.show()

In [ ]:
summary = df_long.groupby('station')['value'].describe()
print(summary)

In [ ]:
df_long

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
for station, group in df_long.groupby('station'):
    plt.plot(group['datetime'], group['value'], label=station)
plt.legend()
plt.title("Pollutant Concentration Over Time")
plt.ylabel("µg/m³")
plt.xticks(rotation=135)
plt.show()

In [ ]:
df_long = df_long.sort_values('datetime', ascending=True)
df_long

In [ ]:
# transform() keeps the result aligned to df_long's original index, so it
# can be assigned straight back as a new column, unlike groupby().apply()
df_long['rolling_mean'] = df_long.groupby('station')['value'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
df_long

In [ ]:
plt.figure(figsize=(12, 5))
sns.lineplot(data=df_long, x='datetime', y='rolling_mean', hue='station')
plt.legend()
plt.title("Pollutant Concentration Over Time")
plt.ylabel("µg/m³")
plt.xticks(rotation=45)
plt.show()

In [ ]:
### Plot just one station
import matplotlib.pyplot as plt
import seaborn as sns

# Filter only one station
df_paris = df_long[df_long['station'].str.contains('paris', case=False)]

# Create a figure with 1 row and 2 columns (side by side)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# --- Left plot: raw data ---
axes[0].plot(df_paris['datetime'], df_paris['value'], color='steelblue')
axes[0].set_title("Raw Pollutant Concentration (Paris)")
axes[0].set_ylabel("µg/m³")
axes[0].set_xlabel("Datetime")
axes[0].tick_params(axis='x', rotation=45)

# --- Right plot: rolling mean (smoothed) ---
sns.lineplot(data=df_paris, x='datetime', y='rolling_mean', color='darkred', ax=axes[1])
axes[1].set_title("Smoothed (Rolling Mean) Concentration (Paris)")
axes[1].set_ylabel("µg/m³")
axes[1].set_xlabel("Datetime")
axes[1].tick_params(axis='x', rotation=45)

# Adjust layout
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create a figure with 1 row and 2 columns (side by side)
fig, axes = plt.subplots(2, 2, figsize=(14, 5), sharey=True)

# --- Left plot: raw data per station ---
for station, group in df_long.groupby('station'):
    axes[0].plot(group['datetime'], group['value'], label=station)
axes[0].set_title("Raw Pollutant Concentration Over Time")
axes[0].set_ylabel("µg/m³")
axes[0].set_xlabel("Datetime")
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend()

# --- Right plot: rolling mean (smoothed) ---
sns.lineplot(data=df_long, x='datetime', y='rolling_mean', hue='station', ax=axes[1])
axes[1].set_title("Smoothed (Rolling Mean) Concentration")
axes[1].set_ylabel("µg/m³")
axes[1].set_xlabel("Datetime")
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend()

# Adjust layout to avoid overlap
plt.tight_layout()
plt.show()

In [ ]:
df_aq

In [ ]:
wide = df_aq.copy()
wide

In [ ]:
wide['datetime'] = pd.to_datetime(wide['datetime'])
wide.dtypes

In [ ]:
wide.rename(columns={'station_antwerp': 'Antwerp', 'station_paris': 'Paris', 'station_london':'London'}, inplace=True)
print(wide)

In [ ]:
sns.heatmap(wide.corr(), annot=True, cmap='coolwarm')

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# Drop rows with missing data for both cities
subset = wide[['London', 'Paris']].dropna()

x = subset['London']
y = subset['Paris']

# Pearson correlation
r, p = pearsonr(x, y)

# Create plot
fig, ax = plt.subplots(figsize=(6,5))
sns.regplot(
    data=subset,
    x='London', y='Paris',
    color='red',
    ci=99,
    scatter_kws={'alpha':0.7, 'color':'black'},
    ax=ax
)

# Titles and labels
ax.set_title('London vs Paris Pollutant Levels (with Trend Line)')
ax.set_xlabel('London (µg/m³)')
ax.set_ylabel('Paris (µg/m³)')

# Annotate correlation in **top-right corner**
ax.text(
    0.97, 0.95,
    f"Pearson r = {r:.2f}\n(p = {p:.3f})",
    transform=ax.transAxes,            # use Axes transform (not data coords)
    fontsize=10,
    verticalalignment='top',
    horizontalalignment='right',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='red', alpha=0.4)
)

plt.tight_layout()
plt.show()

In [ ]:
wide

In [ ]:
# This will raise an error: wide's index is still the default integer row index,
# not the datetime column, so converting the index itself doesn't help.
wide.index = pd.to_datetime(wide.index)  # doesn't fix it, see the cell below
daily = wide.resample('D').mean()
daily.plot()
plt.title("Daily Mean Concentrations")

In [ ]:
# Set the datetime column as the index (not the integer row index)
wide = wide.set_index('datetime').sort_index()
wide

In [ ]:
# Daily resample (numeric columns only) and drop all-NaN days
daily = wide.resample('D').mean(numeric_only=True).dropna(how='all')
daily

In [ ]:
# Plot
ax = daily.plot(figsize=(10,4))
ax.set_title("Daily Mean Concentrations")
ax.set_xlabel("Date"); ax.set_ylabel("Mean concentration")
for spine in ("top","right"): ax.spines[spine].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# Sanity checks
print("Index span:", daily.index.min(), "changed to", daily.index.max())
print("Daily rows:", len(daily))

In [ ]:
df_long.groupby('station')['value'].describe()

In [ ]:
from scipy.stats import f_oneway

# Extract NO concentrations for each country
antwerp = df_long.loc[df_long['station'].str.contains('antwerp', case=False), 'value'].dropna()
paris   = df_long.loc[df_long['station'].str.contains('paris', case=False), 'value'].dropna()
london  = df_long.loc[df_long['station'].str.contains('london', case=False), 'value'].dropna()

# Perform one-way ANOVA
f_stat, p_val = f_oneway(antwerp, paris, london)

print(f"F-statistic: {f_stat:.3f}, p-value: {p_val:.5f}")

## Comparing stations with a multiple linear regression

We already tested whether the three stations differ using a one-way ANOVA. A multiple linear regression can answer the same question, but it also gives us more to work with.


Specifically, it lets us:
- Quantify how much higher or lower each city is relative to a baseline.
- Get regression coefficients with confidence intervals.
- Add more predictors later, such as hour of day or temperature.


In [ ]:
df_no = df_long[['station', 'value']].dropna().copy()
df_no['station'] = df_no['station'].str.capitalize()
df_no.head()

### Fit the model using statsmodels

We will use ordinary least squares (OLS).

`C()` is statsmodels' formula syntax for telling the model that `station` is a categorical variable, that is, Paris, London and Antwerp as distinct groups, not a numeric quantity.

Without the `C()`, statsmodels would try to treat station as numeric, which would make no sense here.


In [ ]:
import statsmodels.formula.api as smf

# Encode the model: value ~ station
model = smf.ols('value ~ C(station)', data=df_no).fit()

# Print regression summary
print(model.summary())

### Interpretation
- One station (alphabetically first, e.g. Antwerp) is used as the baseline.
- Coefficients for Paris and London show how much their mean differs from Antwerp.
- The p-values test if each coefficient ≠ 0 → equivalent to pairwise ANOVA comparisons.
- The F-statistic at the bottom of the summary is identical to your ANOVA test.

### Notice that Antwerp is not "there"
When you include a categorical variable like `station` in a regression, statsmodels automatically creates dummy (indicator) variables for each category, but it drops one category to avoid the "dummy variable trap" (perfect multicollinearity). That dropped category becomes the baseline, or reference group.

In this case we have three categories: Antwerp, London and Paris. statsmodels encoded them as follows:
- Antwerp is the baseline, shown as the Intercept.
- London is `C(station)[T.London]`.
- Paris is `C(station)[T.Paris]`.

So Antwerp is the reference, and Paris and London are both compared against it.


#### Comparing everything against Paris instead
If you would prefer to compare everything to Paris rather than Antwerp, you can tell statsmodels explicitly which category to treat as the baseline.

This is where the `Treatment(reference="Paris")` argument comes in.


In [ ]:
# Treatment(reference="Paris") overrides statsmodels' default (alphabetical) baseline
model_v_paris = smf.ols('value ~ C(station, Treatment(reference="Paris"))', data=df_no).fit()
print(model_v_paris.summary())

In [ ]:
## Adding multiple variables
df_no['datetime'] = pd.to_datetime(df_long['datetime'])
df_no['hour'] = df_no['datetime'].dt.hour
print(df_no)

# Model: NO ~ station + hour
model2 = smf.ols('value ~ C(station) + hour', data=df_no).fit()  # adding hour as a second predictor
print(model2.summary())


### Mini-Assignment (Independent Practice)
- Count missing values for each station.
- Fill missing with **station-wise mean**.
- Compute **24-hour rolling standard deviation** to detect sensor variability.
- Plot NO₂ levels alongside rolling std for at least **two** stations.

Reflect on:
- Which station has the most missing data?
- How does imputation affect trends?
- Which station shows the most noise?
- Any limitations/assumptions? (e.g., missing not at random)



---
# Part 4: Preprocessing, Skew, Binning, Correlation Filters, and Data Integration

We now switch gears from time series back to cross-sectional data. We return to the penguins dataset to demonstrate skew correction, normalisation, and standardisation, techniques you will use constantly to get variables into a shape that statistical models and machine learning algorithms can handle sensibly.

Then we build synthetic water-quality tables from scratch to practice the different ways of combining datasets: merging, concatenating, and joining. This is one of the most common real-world tasks in environmental data analysis, since your sample measurements, site metadata, and lab results usually arrive in separate files and need to be brought together before you can analyse anything.



In [ ]:
import os
os.getcwd()

In [ ]:
# Reuse df_penguins_clean created earlier (run the earlier Part 1 cells first if needed)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

df_penguins_clean = pd.read_csv("Penguins_data.csv")

# Simulate a right-skewed variable (e.g., foraging time in hours)
np.random.seed(42)
df_penguins_clean['foraging_time'] = np.random.exponential(scale=2.0, size=len(df_penguins_clean))

print("Skewness (foraging_time):", df_penguins_clean['foraging_time'].skew())


In [ ]:
df_penguins_clean

In [ ]:
# Visualise original distribution
plt.figure()
sns.histplot(df_penguins_clean['foraging_time'], kde=True)
plt.title("Simulated Foraging Time (Right-Skewed)")
plt.xlabel("Time (hrs)")
plt.show()

In [ ]:
# Log-transform to reduce skew (log1p handles zeros safely)
df_penguins_clean['log_foraging_time'] = np.log1p(df_penguins_clean['foraging_time'])

plt.figure()
sns.histplot(df_penguins_clean['log_foraging_time'], kde=True)
plt.title("Log-Transformed Foraging Time")
plt.xlabel("log(1 + time)")
plt.show()

In [ ]:
# Normalisation & standardisation examples
from sklearn.preprocessing import MinMaxScaler, StandardScaler

mm = MinMaxScaler()
df_penguins_clean['norm_mass'] = mm.fit_transform(df_penguins_clean[['body_mass_g']])

ss = StandardScaler()
df_penguins_clean['std_flipper'] = ss.fit_transform(df_penguins_clean[['flipper_length_mm']])

df_penguins_clean[['body_mass_g','norm_mass','flipper_length_mm','std_flipper']].head()


In [ ]:
# Binning and binary discretisation
df_penguins_clean['mass_category'] = pd.cut(
    df_penguins_clean['body_mass_g'],
    bins=[0, 3500, 4500, 6000],
    labels=['Light', 'Medium', 'Heavy']
)

df_penguins_clean['is_heavy'] = np.where(df_penguins_clean['body_mass_g'] > 4500, 1, 0)
df_penguins_clean[['body_mass_g','mass_category','is_heavy']].head()


In [ ]:
# Absolute correlation matrix for numeric columns
num_df = df_penguins_clean.select_dtypes(include='number') #Select numeric columns
cor_abs = num_df.corr().abs() #Compute absolute correlation matrix
display(cor_abs)

In [ ]:
# Upper-triangle mask to avoid duplicate checks
upper = cor_abs.where(~np.tril(np.ones(cor_abs.shape)).astype(bool))  # Mask the upper triangle to avoid duplicate correlations
to_drop = [col for col in upper.columns if any(upper[col] > 0.9)] #Identify columns with correlations > 0.9
print("Highly correlated features to drop:", to_drop) 
display(upper.head()), display(to_drop)

## Drop the highly correlated columns
Now that we have identified which columns are highly correlated with something else, we can drop them from the cleaned DataFrame in one line.


In [ ]:
df_penguins_reduced = df_penguins_clean.drop(columns=to_drop) #Drop them from the full dataset
print(f"New shape: {df_penguins_reduced.shape}")
display(df_penguins_reduced.head())


## Data integration: joins, concat, and export
Real datasets rarely arrive as one tidy table. In practice you will have a table of sample measurements, a separate table of site metadata, and maybe a third table from an external source, all linked by a shared identifier such as `sample_id`. Bringing these together correctly, without silently losing or duplicating rows, is a core data management skill.

We will create synthetic samples and metadata tables, then demonstrate `merge`, `concat`, and `join`. Finally we export a cleaned dataset and a metadata dictionary to Excel, which is good practice whenever you hand a dataset to someone else.



In [ ]:
# Synthetic tables
np.random.seed(42)

samples = pd.DataFrame({
    'sample_id': np.arange(1001, 1011),
    'nitrate_mg_l': np.random.uniform(0.5, 6.0, 10).round(2),
    'phosphate_mg_l': np.random.uniform(0.1, 2.5, 10).round(2),
    'read_date': pd.date_range(start="2023-01-01", periods=10, freq='D')
})
samples

In [ ]:
more_samples = pd.DataFrame({
    'sample_id': [1011, 1012, 1013],
    'nitrate_mg_l': [4.1, 2.9, 5.2],
    'phosphate_mg_l': [1.2, 0.8, 1.9],
    'read_date': pd.to_datetime(['2023-01-11', '2023-01-12', '2023-01-13'])
})
more_samples

In [ ]:
metadata = pd.DataFrame({
    'sample_id': [1001, 1002, 1003, 1005, 1007, 1008, 1010, 1011],
    'location': ['Site A','Site B','Site C','Site A','Site D','Site B','Site E','Site F'],
    'site_type': ['River','Estuary','Lake','River','Estuary','Lake','River','River'],
    'sample_depth_m': np.random.choice([1.0, 2.5, 5.0], size=8)
})
metadata

In [ ]:
# Merge examples
merged_inner = pd.merge(samples, metadata, on='sample_id', how='inner')
merged_inner

In [ ]:
merged_left  = pd.merge(samples, metadata, on='sample_id', how='left')
merged_left

In [ ]:
merged_right = pd.merge(samples, metadata, on='sample_id', how='right')
merged_right

In [ ]:
merged_outer = pd.merge(samples, metadata, on='sample_id', how='outer')
merged_outer

In [ ]:
print("Inner:", merged_inner.shape, "Left:", merged_left.shape, "Right:", merged_right.shape, "Outer:", merged_outer.shape)
display(merged_left.head())


A join combines two tables (DataFrames in pandas) based on a shared column, often called a key. Examples include `sample_id`, `species_id`, or `country_code`.

- **Inner join**: keeps only rows where the key exists in both tables.
- **Left join**: keeps all rows from the left table (`samples`) and matches whatever exists on the right.
- **Right join**: keeps all rows from the right table (`metadata`) and matches whatever exists on the left.
- **Outer join**: keeps everything from both tables, filling in missing matches with `NaN`.


In [ ]:
# Concat examples
samples_combined = pd.concat([samples, more_samples], ignore_index=True)
samples_combined


In [ ]:
extra_info = pd.DataFrame({
    'qc_flag': ['OK'] * len(samples_combined),
    'instrument_id': np.random.choice(['X1','X2'], size=len(samples_combined))
}, index=samples_combined.index)
extra_info

In [ ]:
samples_with_qc = pd.concat([samples_combined, extra_info], axis=1)

display(samples_combined.tail())
display(samples_with_qc.tail())

In [ ]:
# Join on index
samples_indexed  = samples.set_index('sample_id')
metadata_indexed = metadata.set_index('sample_id')
display(samples_indexed)
display(metadata_indexed)

In [ ]:
joined = samples_indexed.join(metadata_indexed, how='left')
joined.head()

In [ ]:
# Merge external metadata and export to Excel (two sheets)
external_meta = pd.DataFrame({
    'sample_id': [1001, 1003, 1006, 1011],
    'land_use': ['Agricultural','Urban','Forest','Wetland'],
    'type': ['River','River','Stream','Lake']
})

df_full = pd.merge(samples_combined, external_meta, on='sample_id', how='left')

In [ ]:
display(external_meta)
display(df_full)

In [ ]:
# Metadata dictionary
meta_dict = {
    'sample_id': 'Unique sample identifier',
    'nitrate_mg_l': 'Nitrate concentration in mg/L',
    'phosphate_mg_l': 'Phosphate concentration in mg/L',
    'read_date': 'Date sample was read',
    'location': 'Sample collection site',
    'site_type': 'Type of site (River, Estuary, etc.)',
    'sample_depth_m': 'Depth of sampling in meters',
    'qc_flag': 'Quality control flag for the record',
    'instrument_id': 'Instrument identifier',
    'land_use': 'Dominant land use near site',
    'type': 'Hydrological type (river, stream, lake, etc.)'
}
dict_df = pd.DataFrame(list(meta_dict.items()), columns=['Column','Description'])
display(dict_df)

In [ ]:
import openpyxl
# Write Excel file
out_xlsx = "synthetic_water_quality.xlsx"
with pd.ExcelWriter(out_xlsx) as writer:
    df_full.to_excel(writer, sheet_name="Cleaned Data", index=False)
    dict_df.to_excel(writer, sheet_name="Metadata Dictionary", index=False)

out_xlsx


---
## Practice (Suggested)
1. Recreate the **IQR** and **z-score** outlier flags for **flipper_length_mm** and compare lists.  
2. Build a method-chained pipeline that: drops NAs → adds `bill_ratio` → groups by species & sex → computes mean `bill_ratio`.  
3. For air quality, compare **two** different stations’ hourly profiles and comment on similarities/differences.  
4. Add a new synthetic table (e.g., instrument calibration dates) and demonstrate a **left join** into `samples_with_qc`.



### Notes
- Always **log** imputation and flagging decisions (reproducibility!).  
- Prefer **transparent** preprocessing with code + comments over hidden spreadsheet steps.  
- Keep raw data read-only, and create a separate **clean** dataset for analysis and export.
